# Splits

In [11]:
import torch

data = torch.load("/home/ubuntu/genome/id_representations.pt", map_location="cpu")

embeddings = data["representations"]
image_ids = data["image_ids"]

print("Total samples:", len(embeddings))


N = 15000

indices = torch.randperm(len(embeddings))[:N]

embeddings = embeddings[indices]
image_ids = [image_ids[i] for i in indices.tolist()]


train_size = 12000
val_size = 1500
test_size = 1500

train_emb = embeddings[:train_size]
val_emb = embeddings[train_size : train_size + val_size]
test_emb = embeddings[train_size + val_size : train_size + val_size + test_size]

train_paths = image_ids[:train_size]
val_paths = image_ids[train_size : train_size + val_size]
test_paths = image_ids[train_size + val_size : train_size + val_size + test_size]


torch.save({"embeddings": train_emb, "paths": train_paths}, "genome_train.pt")

torch.save({"embeddings": val_emb, "paths": val_paths}, "genome_val.pt")

torch.save({"embeddings": test_emb, "paths": test_paths}, "genome_test.pt")

print("Saved splits:")
print("train:", train_emb.shape)
print("val:", val_emb.shape)
print("test:", test_emb.shape)

Total samples: 15000
Saved splits:
train: torch.Size([12000, 2048])
val: torch.Size([1500, 2048])
test: torch.Size([1500, 2048])


# OoD Split

In [7]:
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

data = torch.load("genome_test.pt", map_location="cpu")

embeddings = data["embeddings"].numpy()
image_ids = data["paths"]

print("Total samples:", len(embeddings))

# Expecting 2000 samples
assert len(embeddings) >= 1500


norm_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)


num_batches = 15
batch_size = 100

remaining = np.arange(len(norm_embeddings))

rng = np.random.default_rng(42)

batches = []
batch_paths = []
batch_indices = []

for batch_id in range(num_batches):

    print(
        f"Building batch {batch_id + 1}/{num_batches} " f"(remaining={len(remaining)})"
    )

    # Pick a random seed from remaining samples
    seed = rng.choice(remaining)

    # Similarity between seed and remaining samples
    sims = norm_embeddings[remaining] @ norm_embeddings[seed]

    # Get top batch_size most similar samples
    top_local = np.argsort(-sims)[:batch_size]

    selected = remaining[top_local]

    # Store batch
    batches.append(torch.from_numpy(embeddings[selected]).float())

    batch_paths.append([image_ids[i] for i in selected])

    batch_indices.append(selected.tolist())

    # Remove selected samples
    mask = np.ones(len(remaining), dtype=bool)
    mask[top_local] = False
    remaining = remaining[mask]


assert len(batches) == 15

for b in batches:
    assert len(b) == 100

all_assigned = np.concatenate(batch_indices)

assert len(all_assigned) == 1500
assert len(np.unique(all_assigned)) == 1500

print("All checks passed.")
print(f"Created {len(batches)} batches.")
print(f"Batch size: {len(batches[0])}")


torch.save(
    {
        "embeddings": batches,
        "paths": batch_paths,
        "indices": batch_indices,
    },
    "genome_test_similar_batches.pt",
)

print("Saved genome_test_similar_batches.pt")

Total samples: 1500
Building batch 1/15 (remaining=1500)
Building batch 2/15 (remaining=1400)
Building batch 3/15 (remaining=1300)
Building batch 4/15 (remaining=1200)
Building batch 5/15 (remaining=1100)
Building batch 6/15 (remaining=1000)
Building batch 7/15 (remaining=900)
Building batch 8/15 (remaining=800)
Building batch 9/15 (remaining=700)
Building batch 10/15 (remaining=600)
Building batch 11/15 (remaining=500)
Building batch 12/15 (remaining=400)
Building batch 13/15 (remaining=300)
Building batch 14/15 (remaining=200)
Building batch 15/15 (remaining=100)
All checks passed.
Created 15 batches.
Batch size: 100
Saved genome_test_similar_batches.pt
